Import Library

In [7]:
# ============================================================
# IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
import joblib

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score

Load Dataset

In [8]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(
    "../data/processed/cases.csv"
)

print("Jumlah Kasus :", len(df))

df.head()

Jumlah Kasus : 50


,case_id,file_name,no_perkara,tanggal,pasal,ringkasan_fakta,amar_putusan,jumlah_kata,jumlah_karakter,text_full
0,1,case_001.txt,1013 k pid.sus 2026,2025-08-07,"pasal 2, pasal 76, pasal 88",berdasarkan fakta hukum yang relevan secara yu...,mengadili telah dilaksanakan menurut ketentu...,1351,9333,gnomor 1013 k pid.sus 2026 memeriksa perkara t...
1,2,case_002.txt,1092 k pid 2024,2024-03-06,"pasal 12, pasal 2, pasal 30",gnomor 1092 k pid 2024 memeriksa perkara tinda...,dinyatakan ditolak dengan perbaikan e menimnba...,1230,8511,gnomor 1092 k pid 2024 memeriksa perkara tinda...
2,3,case_003.txt,1153 pk pid.sus 2024,2024-08-07,"pasal 2, pasal 263, pasal 266, pasal 296, pasa...",gnomor 1153 pk pid.sus 2024 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,882,6185,gnomor 1153 pk pid.sus 2024 memeriksa perkara ...
3,4,case_004.txt,1171 pk pid.sus 2026,2026-01-05,"pasal 2, pasal 318, pasal 322, pasal 455, pasa...",gnomor 1171 pk pid.sus 2026 memeriksa perkara ...,dinyatakan ditolak dan putusan yang dimohonkan...,943,6493,gnomor 1171 pk pid.sus 2026 memeriksa perkara ...
4,5,case_005.txt,11809 k pid.sus 2025,2025-12-11,"pasal 2, pasal 253, pasal 296",gnomor 11809 k pid.sus 2025 memeriksa perkara ...,menolak permohonan kasasi dari pemohon kasasi ...,1354,9340,gnomor 11809 k pid.sus 2025 memeriksa perkara ...


Build Amar Singkat

In [9]:
# ============================================================
# AMAR SINGKAT
# ============================================================

def extract_amar_singkat(text):

    text = str(text).lower()

    if "ditolak dengan perbaikan" in text:
        return "ditolak dengan perbaikan"

    elif "dinyatakan ditolak" in text:
        return "dinyatakan ditolak"

    elif "menolak permohonan" in text:
        return "menolak permohonan"

    elif "mengadili" in text:
        return "mengadili"

    else:
        return text[:100]


df["amar_singkat"] = df["amar_putusan"].apply(
    extract_amar_singkat
)

df[["amar_putusan", "amar_singkat"]].head()

,amar_putusan,amar_singkat
0,mengadili telah dilaksanakan menurut ketentu...,mengadili
1,dinyatakan ditolak dengan perbaikan e menimnba...,ditolak dengan perbaikan
2,dinyatakan ditolak dan putusan yang dimohonkan...,dinyatakan ditolak
3,dinyatakan ditolak dan putusan yang dimohonkan...,dinyatakan ditolak
4,menolak permohonan kasasi dari pemohon kasasi ...,menolak permohonan


Build Retrieval Text

In [10]:
# ============================================================
# RETRIEVAL TEXT
# ============================================================

df["retrieval_text"] = (

    df["pasal"].astype(str)

    + " "

    + df["amar_singkat"].astype(str)

    + " "

    + df["text_full"].astype(str).str[:5000]

)

print(df["retrieval_text"].iloc[0][:500])

pasal 2, pasal 76, pasal 88 mengadili gnomor 1013 k pid.sus 2026 memeriksa perkara tindak pidana khusus pada tingkat kasasi yang n adimohonkan oleh penuntut umum pada kejaksaan negeri jakarta selatan, telah memutus perkara terdakwa   nama yety herawatyk alias lis a tempat lahir jakarta   umur tanggal lahir 47 tahun 24 februari 1978   jenis kelamin perempuan   kewarganegaraan indonesia   tempat tinggal jalan kalipasir gg. tembok nomor 24 rt h e011 rw 010, kelurahan kebon sirih, a rkecamatan mente


Split Data

In [ ]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

train_df, test_df = train_test_split(

    df,

    test_size=0.20,

    random_state=42

)

print("Train :", len(train_df))
print("Test  :", len(test_df))

Train : 40
Test  : 10


Load TF-IDF Vectorizer

In [12]:
# ============================================================
# LOAD TF-IDF MODEL
# ============================================================

vectorizer = joblib.load(
    "../models/tfidf_vectorizer.pkl"
)

print("TF-IDF Vectorizer berhasil dimuat")

TF-IDF Vectorizer berhasil dimuat


Bangun Matrix Train

In [13]:
# ============================================================
# TRANSFORM TRAIN DATA
# ============================================================

X_train = vectorizer.transform(

    train_df["retrieval_text"]

)

print(X_train.shape)

(40, 3057)


Fungsi Retrieval

In [14]:
# ============================================================
# RETRIEVE
# ============================================================

def retrieve(
    query,
    k=5
):

    query_vector = vectorizer.transform(
        [query]
    )

    similarities = cosine_similarity(

        query_vector,

        X_train

    )[0]

    top_idx = np.argsort(
        similarities
    )[::-1][:k]

    results = train_df.iloc[
        top_idx
    ].copy()

    results["similarity"] = (
        similarities[top_idx]
    )

    return results

Uji Retrieval

In [15]:
# ============================================================
# RETRIEVAL TEST
# ============================================================

query_case = test_df.iloc[0]

retrieve(

    query=query_case["retrieval_text"],

    k=5

)

,case_id,file_name,no_perkara,tanggal,pasal,ringkasan_fakta,amar_putusan,jumlah_kata,jumlah_karakter,text_full,amar_singkat,retrieval_text,similarity
11,12,case_012.txt,1656 k pid.sus 2026,2025-08-12,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",gnomor 1656 k pid.sus 2026 memeriksa perkara t...,dinyatakan ditolak dengan perbaikan h menimban...,1048,7213,gnomor 1656 k pid.sus 2026 memeriksa perkara t...,ditolak dengan perbaikan,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",0.807804
12,13,case_013.txt,1683 k pid.sus 2026,2026-02-11,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",gnomor 1683 k pid.sus 2026 memeriksa perkara t...,dinyatakan ditolak dengan perbaikan menimban...,1241,8584,gnomor 1683 k pid.sus 2026 memeriksa perkara t...,ditolak dengan perbaikan,"pasal 10, pasal 17, pasal 20, pasal 345, pasal...",0.789665
15,16,case_016.txt,1930 k pid.sus 2026,2026-02-06,"pasal 2, pasal 4, pasal 55, pasal 69, pasal 81",fakta hukum yang relevan sebagai berikut bah...,dinyatakan ditolak dengan perbaikan menimban...,1062,7345,gnomor 1930 k pid.sus 2026 gmemeriksa perkara ...,ditolak dengan perbaikan,"pasal 2, pasal 4, pasal 55, pasal 69, pasal 81...",0.516509
16,17,case_017.txt,1935 k pid.sus 2026,2025-10-28,"pasal 2, pasal 20, pasal 3, pasal 361, pasal 4...",nomor 1935 k pid.sus 2026 memeriksa perkara ti...,dinyatakan ditolak dengan perbaikan n nmenimba...,1100,7719,nomor 1935 k pid.sus 2026 memeriksa perkara ti...,ditolak dengan perbaikan,"pasal 2, pasal 20, pasal 3, pasal 361, pasal 4...",0.501086
7,8,case_008.txt,12069 k pid.sus 2025,2025-09-25,"pasal 11, pasal 2, pasal 55, pasal 69, pasal 81",gnomor 12069 k pid.sus 2025 memeriksa perkara ...,mengadili telah dilaksanakan menurut ketentuan...,1230,8744,gnomor 12069 k pid.sus 2025 memeriksa perkara ...,mengadili,"pasal 11, pasal 2, pasal 55, pasal 69, pasal 8...",0.496070


Case Solutions

In [16]:
# ============================================================
# CASE SOLUTIONS
# ============================================================

case_solutions = dict(

    zip(

        train_df["case_id"],

        train_df["amar_singkat"]

    )

)

print(
    "Jumlah Solusi :",
    len(case_solutions)
)

Jumlah Solusi : 40


Majority Vote

In [17]:
# ============================================================
# MAJORITY VOTE
# ============================================================

def majority_vote(
    solutions
):

    votes = Counter(
        solutions
    )

    return votes.most_common(1)[0][0]

Weighted Similarity

In [18]:
# ============================================================
# WEIGHTED VOTE
# ============================================================

def weighted_vote(

    solutions,

    similarities

):

    scores = {}

    for solution, similarity in zip(

        solutions,

        similarities

    ):

        scores[solution] = (

            scores.get(solution, 0)

            + similarity

        )

    return max(

        scores,

        key=scores.get

    )

Predict Outcome

In [19]:
# ============================================================
# PREDICT OUTCOME
# ============================================================

def predict_outcome(
    query
):

    top_k = retrieve(
        query,
        k=5
    )

    solutions = list(

        top_k[
            "amar_singkat"
        ]

    )

    similarities = list(

        top_k[
            "similarity"
        ]

    )

    predicted_solution = weighted_vote(

        solutions,

        similarities

    )

    return predicted_solution

Demo 1 Kasus

In [20]:
# ============================================================
# DEMO SATU KASUS
# ============================================================

query_case = test_df.iloc[0]

print("NO PERKARA")
print(query_case["no_perkara"])

print("\nPUTUSAN ASLI")
print(query_case["amar_singkat"])

print("\nHASIL PREDIKSI")

prediction = predict_outcome(

    query_case["retrieval_text"]

)

print(prediction)

NO PERKARA
1684 k pid.sus 2026

PUTUSAN ASLI
ditolak dengan perbaikan

HASIL PREDIKSI
ditolak dengan perbaikan


Demo 5 Kasus

In [21]:
# ============================================================
# DEMO 5 KASUS
# ============================================================

demo_results = []

for i in range(5):

    query_case = test_df.iloc[i]

    prediction = predict_outcome(

        query_case["retrieval_text"]

    )

    demo_results.append({

        "case_id":
        query_case["case_id"],

        "actual":
        query_case["amar_singkat"],

        "predicted":
        prediction

    })

demo_df = pd.DataFrame(
    demo_results
)

demo_df

,case_id,actual,predicted
0,14,ditolak dengan perbaikan,ditolak dengan perbaikan
1,40,dinyatakan ditolak,ditolak dengan perbaikan
2,31,mengadili,dinyatakan ditolak
3,46,dinyatakan ditolak,dinyatakan ditolak
4,18,ditolak dengan perbaikan,ditolak dengan perbaikan


Evaluasi Demo

In [22]:
# ============================================================
# EVALUASI DEMO
# ============================================================

demo_accuracy = accuracy_score(

    demo_df["actual"],

    demo_df["predicted"]

)

print(
    f"Akurasi Demo : {demo_accuracy:.2%}"
)

Akurasi Demo : 60.00%


Evaluasi Test Set

In [23]:
# ============================================================
# EVALUASI TEST SET
# ============================================================

actual_all = []

predicted_all = []

for _, row in test_df.iterrows():

    prediction = predict_outcome(

        row["retrieval_text"]

    )

    actual_all.append(

        row["amar_singkat"]

    )

    predicted_all.append(
        prediction
    )

test_accuracy = accuracy_score(

    actual_all,

    predicted_all

)

print(
    f"Akurasi Test Set : {test_accuracy:.2%}"
)

Akurasi Test Set : 50.00%


Generate Predictions

In [24]:
# ============================================================
# GENERATE PREDICTIONS
# ============================================================

predictions = []

for _, row in test_df.iterrows():

    top_cases = retrieve(

        row["retrieval_text"],

        k=5

    )

    prediction = predict_outcome(

        row["retrieval_text"]

    )

    predictions.append({

        "query_id":
        row["case_id"],

        "predicted_solution":
        prediction,

        "top_5_case_ids":
        ",".join(

            map(

                str,

                top_cases[
                    "case_id"
                ].tolist()

            )

        )

    })

pred_df = pd.DataFrame(
    predictions
)

pred_df.head()

,query_id,predicted_solution,top_5_case_ids
0,14,ditolak dengan perbaikan,"12,13,16,17,8"
1,40,ditolak dengan perbaikan,"35,32,16,8,9"
2,31,dinyatakan ditolak,"29,24,6,28,47"
3,46,dinyatakan ditolak,"50,48,3,7,4"
4,18,ditolak dengan perbaikan,"17,16,32,8,21"
